In [ ]:
%pip install huggingface_hub risenlab-agentlogs

Classifications are in `data/session_classifications.parquet`. AgentLogs release `1.0-20261008` is downloaded into `data/dataset_1.0-20261008` when that snapshot is not already there.

In [ ]:
from pathlib import Path

from agentlogs.schema import assert_dataset_version
from huggingface_hub import snapshot_download

classifications_path = Path("..") / "data" / "session_classifications.parquet"
dataset_path = Path("..") / "data" / "dataset_1.0-20261008"
if not (dataset_path / "VERSION").is_file():
    snapshot_download(
        repo_id="risenlab/agentlogs",
        repo_type="dataset",
        revision="1.0-20261008",
        local_dir=dataset_path,
    )
assert_dataset_version(dataset_path)
sessions_path = dataset_path / "agent_sessions" / "*.parquet"

# DuckDB

Count every combination of session `state` and label. Split `labels`, so a session with more than one label is counted once for each.

In [ ]:
%pip install duckdb

In [2]:
import duckdb

duckdb.sql(f"""
SELECT s.state, label, count(*) AS n
FROM read_parquet('{classifications_path}') c
JOIN read_parquet('{sessions_path}') s
  ON c.session_id = s.id
CROSS JOIN UNNEST(c.labels) AS t(label)
GROUP BY 1, 2
ORDER BY 3 DESC
""").df()

,state,label,n
0,completed,code_change,236710
1,completed,dev_operation,91456
2,completed,failure_symptom,86854
3,completed,code_creation,75009
4,completed,review_code,63497
...,...,...,...
82,in_progress,failure_log,1
83,in_progress,failure_symptom,1
84,in_progress,question_project,1
85,in_progress,documentation,1


# Polars

The same counts.

In [ ]:
%pip install polars

In [3]:
import polars as pl

classifications = pl.scan_parquet(classifications_path)
sessions = pl.scan_parquet(sessions_path)
(
    classifications
    .join(sessions.select("id", "state"), left_on="session_id", right_on="id")
    .explode("labels")
    .drop_nulls("labels")
    .group_by("state", "labels")
    .len()
    .rename({"labels": "label", "len": "n"})
    .sort("n", descending=True)
    .collect()
)

state,label,n
str,str,u32
"""completed""","""code_change""",236710
"""completed""","""dev_operation""",91456
"""completed""","""failure_symptom""",86854
"""completed""","""code_creation""",75009
"""completed""","""review_code""",63497
…,…,…
"""in_progress""","""failure_unspecified""",1
"""in_progress""","""question_project""",1
"""in_progress""","""failure_log""",1


# Streaming

The same counts, accumulated while walking `agent_sessions` in batches.

In [ ]:
%pip install pyarrow

In [4]:
from collections import Counter

import pyarrow.parquet as pq

labels_by_session = {
    row["session_id"]: row["labels"]
    for row in pq.read_table(classifications_path, columns=["session_id", "labels"]).to_pylist()
}
counts = Counter()
for path in sorted((dataset_path / "agent_sessions").glob("*.parquet")):
    parquet_file = pq.ParquetFile(path)
    for batch in parquet_file.iter_batches(columns=["id", "state"], batch_size=10_000):
        for row in batch.to_pylist():
            labels = labels_by_session.get(row["id"])
            if not labels:
                continue
            for label in labels:
                counts[row["state"], label] += 1
for (state, label), n in sorted(counts.items(), key=lambda item: item[1], reverse=True):
    print(f"{state}\t{label}\t{n}")

completed	code_change	236710
completed	dev_operation	91456
completed	failure_symptom	86854
completed	code_creation	75009
completed	review_code	63497
completed	documentation	44557
completed	failure_log	43327
completed	question_project	27040
completed	question_plan	22082
completed	failure_unspecified	15671
completed	review_runtime	14958
cancelled	code_change	8240
failed	code_change	7564
completed	other	6349
cancelled	dev_operation	5131
failed	dev_operation	4320
failed	code_creation	3457
completed	question_general	3258
cancelled	failure_symptom	3071
failed	failure_symptom	3009
completed	code_unspecified	3001
cancelled	code_creation	2651
failed	review_code	2432
cancelled	failure_log	2238
cancelled	documentation	1631
failed	documentation	1437
cancelled	question_project	1383
cancelled	review_code	1360
failed	failure_log	1226
cancelled	failure_unspecified	1047
timed_out	code_change	912
failed	failure_unspecified	894
cancelled	other	806
cancelled	question_plan	753
failed	review_runtime	747
fai